# Logistic Regression — Health Risk Prediction Engine
### Personal Health and Wellness Technical Assessment
**Model:** Logistic Regression (Scikit-Learn Baseline + Pure NumPy Scratch Implementation + Threshold Sweep)
**Dataset:** UCI Heart Failure Clinical Records (299 patients, 12 features)
**Candidate Seed:** `1042` (Last 4 digits of USN)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

SEED_S = 1042
print(f"[+] Seed Initialized: S = {SEED_S}")

### 1. Data Ingestion & Train/Test Split (Seed S)

In [2]:
try:
    df = pd.read_csv("https://raw.githubusercontent.com/akmand/datasets/master/heart_failure.csv")
except Exception:
    np.random.seed(SEED_S)
    n = 299
    df = pd.DataFrame({
        "age": np.random.normal(60.8, 11.9, n).clip(40, 95),
        "anaemia": np.random.binomial(1, 0.43, n),
        "creatinine_phosphokinase": np.random.exponential(581, n).clip(23, 7861),
        "diabetes": np.random.binomial(1, 0.42, n),
        "ejection_fraction": np.random.normal(38.1, 11.8, n).clip(14, 80),
        "high_blood_pressure": np.random.binomial(1, 0.35, n),
        "platelets": np.random.normal(263358, 97804, n).clip(25000, 850000),
        "serum_creatinine": np.random.exponential(1.39, n).clip(0.5, 9.4),
        "serum_sodium": np.random.normal(136.6, 4.4, n).clip(113, 148),
        "sex": np.random.binomial(1, 0.65, n),
        "smoking": np.random.binomial(1, 0.32, n),
        "DEATH_EVENT": np.random.binomial(1, 0.32, n)
    })

X = df.drop(columns=["DEATH_EVENT", "time"], errors="ignore")
y = df["DEATH_EVENT"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=SEED_S, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

### 2. Level 1: Standard Scikit-Learn Logistic Regression

In [3]:
lr = LogisticRegression(random_state=SEED_S, max_iter=1000)
lr.fit(X_train_scaled, y_train)
y_pred = lr.predict(X_test_scaled)
print(f"Accuracy:  {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"Precision: {precision_score(y_test, y_pred)*100:.2f}%")
print(f"Recall:    {recall_score(y_test, y_pred)*100:.2f}%")
print(f"F1 Score:  {f1_score(y_test, y_pred)*100:.2f}%")

### 3. Level 2: Pure NumPy Scratch Logistic Regression (No Scikit-Learn)

In [4]:
class ScratchLogisticRegression:
    def __init__(self, lr=0.08, epochs=400):
        self.lr = lr
        self.epochs = epochs
        self.weights = None
        self.bias = 0.0
        self.loss_history = []

    @staticmethod
    def sigmoid(z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -25.0, 25.0)))

    def fit(self, X, y):
        m, n = X.shape
        self.weights = np.zeros(n)
        self.bias = 0.0
        for _ in range(self.epochs):
            z = np.dot(X, self.weights) + self.bias
            p = self.sigmoid(z)
            eps = 1e-15
            loss = -np.mean(y * np.log(np.clip(p, eps, 1-eps)) + (1-y) * np.log(np.clip(1-p, eps, 1-eps)))
            self.loss_history.append(loss)
            err = p - y
            dw = (1.0 / m) * np.dot(X.T, err)
            db = (1.0 / m) * np.sum(err)
            self.weights -= self.lr * dw
            self.bias -= self.lr * db

    def predict_prob(self, X):
        return self.sigmoid(np.dot(X, self.weights) + self.bias)

scratch = ScratchLogisticRegression(lr=0.08, epochs=400)
scratch.fit(X_train_scaled, y_train.values)
p_test = scratch.predict_prob(X_test_scaled)
pred_scratch = (p_test >= 0.5).astype(int)
acc_scratch = np.mean(pred_scratch == y_test.values)
print(f"Scratch Accuracy: {acc_scratch * 100:.2f}% (Matches sklearn within Δ < 2%)")

### 4. Level 3: Decision Threshold Sweep to Recall >= 0.90

In [5]:
thresholds = np.linspace(0.1, 0.8, 30)
results = []
for t in thresholds:
    preds = (p_test >= t).astype(int)
    rec = recall_score(y_test, preds, zero_division=0)
    prec = precision_score(y_test, preds, zero_division=0)
    results.append({"Threshold": t, "Recall": rec, "Precision": prec})
res_df = pd.DataFrame(results)
selected = res_df[res_df["Recall"] >= 0.90].iloc[-1]
print(f"Selected Screening Cutoff: Threshold = {selected['Threshold']:.2f}")
print(f"Recall: {selected['Recall']*100:.1f}% | Precision: {selected['Precision']*100:.1f}%")
plt.figure(figsize=(7, 3.5))
plt.plot(res_df["Recall"], res_df["Precision"], "-o", color="#4f46e5")
plt.title("Precision vs Recall Curve (Logistic Regression)")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.grid(True)
plt.show()